#### Problem Statement:

We are analyzing fuel station operations to identify revenue leakage caused by potential under-dispensing of fuel and low customer retention driven by poor customer experience (e.g., long wait times and trust issues). These issues directly impact revenue, customer lifetime value, and operational efficiency across stations.

#### Core Hypotheses:

🔹1.) Fraud Hypothesis
→ Stations where actual fuel dispensed < expected fuel show higher revenue leakage and lower repeat visits.

🔹2.) Wait Time Hypothesis
→ Higher average wait times (especially peak hours) reduce customer repeat rate.

🔹3.) Trust Hypothesis
→ Customers exposed to suspicious stations (fraud/poor service) are less likely to return within 7 days.

#### Strategic Focus (Important)

We will prioritize:

🔹1.) Fraud detection (money loss)

🔹2.) Wait time (experience)

🔹3.) Retention (outcome metric)

## PHASE 1: SETUP

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

## PHASE 2: DATA CREATION

In [2]:
# 👤 1. CUSTOMERS TABLE

num_customers = 5000

customers = pd.DataFrame({
    "customer_id": range(1, num_customers + 1),
    "customer_type": np.random.choice(
        ["frequent", "occasional"], 
        size=num_customers, 
        p=[0.3, 0.7]
    ),
    "signup_date": pd.to_datetime("2025-01-01") + 
                   pd.to_timedelta(np.random.randint(0, 180, num_customers), unit='D')
})

customers.head()

,customer_id,customer_type,signup_date
0,1,occasional,2025-02-20
1,2,occasional,2025-03-12
2,3,occasional,2025-02-17
3,4,occasional,2025-01-19
4,5,frequent,2025-01-06


In [3]:
# ⛽ 2. STATIONS TABLE

num_stations = 50

stations = pd.DataFrame({
    "station_id": range(1, num_stations + 1),
    "city": np.random.choice(["Mumbai", "Delhi", "Bangalore"], num_stations),
    "num_pumps": np.random.randint(4, 12, num_stations),
    "staff_count": np.random.randint(5, 20, num_stations),
    "quality_score": np.round(np.random.uniform(2.5, 5.0, num_stations), 1),
    "fraud_prone_flag": np.random.choice([0, 1], num_stations, p=[0.8, 0.2])
})

stations.head(10)

,station_id,city,num_pumps,staff_count,quality_score,fraud_prone_flag
0,1,Bangalore,6,18,3.6,0
1,2,Mumbai,6,12,3.2,0
2,3,Mumbai,7,10,3.2,0
3,4,Mumbai,6,16,3.8,0
4,5,Bangalore,9,14,3.6,0
5,6,Mumbai,7,15,3.6,1
6,7,Mumbai,11,17,2.7,0
7,8,Mumbai,10,19,4.3,0
8,9,Bangalore,8,18,3.0,0
9,10,Bangalore,6,9,3.0,1


In [4]:
# 💳 3. TRANSACTIONS TABLE (FULL PIPELINE)

num_transactions = 100000

transactions = pd.DataFrame({
    "transaction_id": range(1, num_transactions + 1),
    "customer_id": np.random.choice(customers["customer_id"], num_transactions),
    "station_id": np.random.choice(stations["station_id"], num_transactions)
})

transactions.head()

,transaction_id,customer_id,station_id
0,1,2583,49
1,2,3838,31
2,3,527,38
3,4,1666,39
4,5,1139,1


In [5]:
# ⏰ TIMESTAMP

def generate_timestamp(n):
    base_date = pd.to_datetime("2025-06-01")
    
    days = np.random.randint(0, 60, n)
    
    hours = np.random.choice(
        [8,9,10,11,17,18,19,20, 12,13,14,15],
        size=n,
        p=[0.1]*8 + [0.05]*4
    )
    
    minutes = np.random.randint(0, 60, n)
    
    return base_date + pd.to_timedelta(days, unit="D") + \
           pd.to_timedelta(hours, unit="h") + \
           pd.to_timedelta(minutes, unit="m")

transactions["timestamp"] = generate_timestamp(num_transactions)

In [6]:
# ⛽ FUEL REQUESTED

transactions["fuel_requested_liters"] = np.round(
    np.random.uniform(5, 50, num_transactions), 2
)

In [7]:
# 🔗 MERGE STATION DATA

transactions = transactions.merge(
    stations[["station_id", "fraud_prone_flag", "num_pumps", "staff_count"]],
    on="station_id",
    how="left"
)

transactions.head()

,transaction_id,customer_id,station_id,timestamp,fuel_requested_liters,fraud_prone_flag,num_pumps,staff_count
0,1,2583,49,2025-06-04 15:15:00,48.04,0,6,15
1,2,3838,31,2025-07-05 11:30:00,48.73,0,4,19
2,3,527,38,2025-06-19 14:00:00,17.86,0,4,18
3,4,1666,39,2025-06-05 10:23:00,36.57,0,5,7
4,5,1139,1,2025-06-06 17:54:00,18.80,0,6,18


In [8]:
# 🚨 FRAUD LOGIC + ROUNDING

def apply_fraud(row):
    if row["fraud_prone_flag"] == 1:
        return row["fuel_requested_liters"] * np.random.uniform(0.95, 0.98)
    else:
        return row["fuel_requested_liters"] * np.random.uniform(0.99, 1.01)

transactions["fuel_dispensed_liters"] = transactions.apply(apply_fraud, axis=1)

# Clean rounding (important)
transactions["fuel_dispensed_liters"] = transactions["fuel_dispensed_liters"].round(2)

In [9]:
# ⏳ WAIT TIME

def calculate_wait(row):
    base_wait = np.random.uniform(2, 5)
    hour = row["timestamp"].hour
    
    if hour in [8,9,10,11,17,18,19,20]:
        peak_factor = np.random.uniform(1.5, 2.5)
    else:
        peak_factor = np.random.uniform(0.8, 1.2)
    
    capacity_factor = (12 - row["num_pumps"]) / 10
    staff_factor = (20 - row["staff_count"]) / 20
    
    wait = base_wait * peak_factor * (1 + capacity_factor + staff_factor)
    
    return round(wait, 2)

transactions["wait_time_minutes"] = transactions.apply(calculate_wait, axis=1)

In [10]:
transactions[["fuel_requested_liters", "fuel_dispensed_liters", "wait_time_minutes"]].head()

,fuel_requested_liters,fuel_dispensed_liters,wait_time_minutes
0,48.04,48.34,4.08
1,48.73,49.12,11.01
2,17.86,17.75,5.82
3,36.57,36.61,16.07
4,18.80,18.64,12.59


In [11]:
# 💰 AMOUNT

price_per_liter = 100

transactions["amount_paid"] = (
    transactions["fuel_dispensed_liters"] * price_per_liter
).round(2)

## PHASE 3: FEATURE ENGINEERING 

In [12]:
# 📊 FRAUD FEATURES

transactions["fuel_diff"] = (
    transactions["fuel_requested_liters"] - 
    transactions["fuel_dispensed_liters"]
)

transactions["fraud_flag"] = transactions["fuel_diff"] > 0.2

In [13]:
# 👥 ADD CUSTOMER TYPE (FINAL CLEAN VERSION)

# Merge only required columns
transactions = transactions.merge(
    customers[["customer_id", "customer_type"]],
    on="customer_id",
    how="left"
)

transactions[["customer_id", "customer_type"]].head()

,customer_id,customer_type
0,2583,occasional
1,3838,frequent
2,527,occasional
3,1666,occasional
4,1139,occasional


In [14]:
# 🔁 RETENTION LOGIC

def assign_repeat_flag(row):
    prob = 0.3
    
    if row["customer_type"] == "frequent":
        prob += 0.3
    
    if row["wait_time_minutes"] > 15:
        prob -= 0.2
    
    if row["fuel_diff"] > 0.2:
        prob -= 0.3
    
    return np.random.rand() < prob

transactions["repeat_customer"] = transactions.apply(assign_repeat_flag, axis=1)

In [15]:
# 🕒 TIME FEATURES

transactions["hour"] = transactions["timestamp"].dt.hour
transactions["day"] = transactions["timestamp"].dt.date

In [16]:
# 🧹 FINAL CLEANUP

transactions = transactions.drop(
    ["fraud_prone_flag", "num_pumps", "staff_count"],
    axis=1
)

In [17]:
transactions.head()
transactions["repeat_customer"].value_counts()

repeat_customer
False    73254
True     26746
Name: count, dtype: int64

In [18]:
# 🧪 SENSOR DATA TABLE

sensor_data = transactions[[
    "transaction_id",
    "fuel_requested_liters",
    "fuel_dispensed_liters"
]].copy()

sensor_data.columns = [
    "transaction_id",
    "expected_fuel",
    "actual_fuel"
]

sensor_data.head()

,transaction_id,expected_fuel,actual_fuel
0,1,48.04,48.34
1,2,48.73,49.12
2,3,17.86,17.75
3,4,36.57,36.61
4,5,18.80,18.64


In [19]:
customers.to_csv("customers.csv", index=False)
stations.to_csv("stations.csv", index=False)
transactions.to_csv("transactions.csv", index=False)
sensor_data.to_csv("sensor_data.csv", index=False)